# Compare Splitting Results Across Parameter Types - NonLinLoc Catalog, Station AXAS2, in the week around the 2015 Axial Seamount Eruption

In [ ]:
# Dependencies

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Updated plotting functions that work with our dataframe structure

def plot_fast_direction_rose(results_df, title="Fast Direction Distribution", 
                              nbins=18, figsize=(8, 8), color='steelblue',
                              edgecolor='black', linewidth=0.5):
    """
    Create a polar rose plot (histogram) of fast directions from splitting results.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' column
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 18 = 10° bins for ±90°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = np.deg2rad(results_df['phi'].values)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (-pi/2 to pi/2 for -90° to +90°)
    bins = np.linspace(-np.pi/2, np.pi/2, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set angular limits (-90° to +90°)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    
    # Set radial ticks
    ax.set_rlabel_position(0)
    
    # Add degree labels
    tick_labels = ['-90°', '-60°', '-30°', '0°', '30°', '60°', '90°']
    tick_positions = np.deg2rad([-90, -60, -30, 0, 30, 60, 90])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    ax.set_ylim(0, 375)
    
    # Add title with statistics
    n_measurements = len(fast_directions)
    # Calculate circular mean for ±90° range
    mean_direction = np.rad2deg(np.arctan2(np.sin(fast_directions).sum(), 
                                           np.cos(fast_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax


def plot_splitting_timeseries_smooth(results_df, station='AXAS2', 
                                     figsize=(14, 8), x_width_days=5, x_overlap=0.95,
                                     y_width_phi=5, y_width_dt=2, y_overlap=0.95,
                                     sigma=2.0, sampling_rate=200.0,
                                     time_column='event_datetime'):
    """
    Create smoothed 2D histogram time-series plots inspired by Baillard's approach.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi', 'dt', and time columns
    station : str
        Station name for title
    figsize : tuple
        Figure size (width, height)
    x_width_days : float
        Width of moving time window in days
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width_phi : float
        Bin width for phi in degrees
    y_width_dt : int
        Bin width for dt in samples (1 sample = 1/sampling_rate seconds)
    y_overlap : float
        Overlap for smoothing in y-direction
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    time_column : str
        Name of the time column in results_df (default 'origin_time')
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.dates import DateFormatter
    import matplotlib.dates as mdates
    from scipy.ndimage import gaussian_filter
    
    # Prepare data from DataFrame
    df = results_df.copy()
    df['time'] = pd.to_datetime(df[time_column])
    df['phi_deg'] = df['phi']
    df['phi_rad'] = np.deg2rad(df['phi'])
    df['phi_rad_norm'] = ((df['phi_rad'] + np.pi/2) % np.pi) - np.pi/2
    df['dt_seconds'] = df['dt']
    df['dt_samples'] = df['dt'] * sampling_rate
    df = df.sort_values('time')
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Convert datetime to matplotlib date numbers
    time_nums = mdates.date2num(df['time'])
    
    # === PHI PLOT (in radians) ===
    # Create bins
    time_range = (time_nums.min(), time_nums.max())
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_days * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))  # Reasonable limits
    
    # Phi bins from -pi/2 to +pi/2 radians (-1.57 to +1.57)
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    n_phi_bins = len(phi_bins) - 1
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_nums, df['phi_rad_norm'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin) - "norm_y=True" in Baillard's code
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    
    # Mask zeros
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot with imshow for smooth appearance
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis with degrees
    ax_phi.set_ylabel('Fast Direction φ (°)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_phi.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Set y-ticks in radians but label with degrees
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    phi_ticks_deg = np.array([-90, -60, -30, 0, 30, 60, 90])
    ax_phi.set_yticks(phi_ticks_rad)
    ax_phi.set_yticklabels([f'{int(d)}°' for d in phi_ticks_deg])
    
    # Add moving average
    window_size = max(5, len(df) // 10)
    if len(df) >= window_size:
        df['phi_rad_ma'] = df['phi_rad_norm'].rolling(window=window_size, center=True).mean()
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'black', linewidth=2,
                   label=f'{window_size}-event moving avg', alpha=0.8)
        ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
                     edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = df['dt_samples'].quantile(0.98)
    # Create bins in samples
    dt_bins = np.arange(0, dt_max_samples + y_width_dt, y_width_dt)
    n_dt_bins = len(dt_bins) - 1
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_nums, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    
    # Mask zeros
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot with imshow
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)
    
    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_dt.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Format dt axis with samples
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, 25)  # Show up to 25 samples (0.125 sec at 200 Hz)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add moving average for dt
    #if len(df) >= window_size:
    #    df['dt_samples_ma'] = df['dt_samples'].rolling(window=window_size, center=True).mean()
    #    ax_dt.plot(df['time'], df['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
    #    ax_dt.plot(df['time'], df['dt_samples_ma'], 'black', linewidth=2,
    #              label=f'{window_size}-event moving avg', alpha=0.8)
    #    ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
    #                edgecolor='white', framealpha=0.7)
    
    # Format x-axis with dates
    date_formatter = DateFormatter('%Y-%m-%d')
    ax_dt.xaxis.set_major_formatter(date_formatter)
    
    # Auto-adjust date locator
    days_span = (df['time'].max() - df['time'].min()).days
    if days_span > 60:
        ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    elif days_span > 14:
        ax_dt.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    else:
        ax_dt.xaxis.set_major_locator(mdates.DayLocator())
    
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics (in both degrees and radians)
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    title = (f"Splitting Parameter Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}° "
             f"({np.deg2rad(phi_mean_deg):.2f} ± {np.deg2rad(phi_std_deg):.2f} rad) | "
             f"δt: {dt_mean_sec:.3f} ± {dt_std_sec:.3f} s "
             f"({dt_mean_sec*sampling_rate:.1f} ± {dt_std_sec*sampling_rate:.1f} samples)")
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df

def plot_fast_direction_rose_eruption_comparison(results_df, eruption_time=None, 
                                                  title_prefix="Fast Direction Distribution",
                                                  nbins=36, figsize=(16, 7), color='steelblue',
                                                  edgecolor='black', linewidth=0.5,
                                                  time_column='event_datetime'):
    """
    Create side-by-side 360° rose plots comparing fast directions before and after eruption.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' and time columns
    eruption_time : UTCDateTime or str
        Time of eruption onset (default: 2015-04-24T06:00:00)
    title_prefix : str
        Prefix for plot titles
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height) for combined plot
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    time_column : str
        Name of the time column in results_df (default 'event_datetime')
    """
    if eruption_time is None:
        eruption_time = UTCDateTime(2015, 4, 24, 6)
    elif not isinstance(eruption_time, UTCDateTime):
        eruption_time = UTCDateTime(eruption_time)
    
    # Convert time column to UTCDateTime for comparison
    df_time = results_df[time_column].apply(lambda x: UTCDateTime(x) if not isinstance(x, UTCDateTime) else x)
    results_before = results_df[df_time < eruption_time]
    results_after = results_df[df_time >= eruption_time]
    
    # Create figure with two subplots
    fig = plt.figure(figsize=figsize)
    
    # Before eruption plot (left)
    ax1 = fig.add_subplot(121, projection='polar')
    plot_rose_subplot(results_before, ax1, f"{title_prefix}\nBefore Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    # After eruption plot (right)
    ax2 = fig.add_subplot(122, projection='polar')
    plot_rose_subplot(results_after, ax2, f"{title_prefix}\nAfter Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    plt.tight_layout()
    return fig, (ax1, ax2), (results_before, results_after)


def plot_rose_subplot(results_df, ax, title, nbins, color, edgecolor, linewidth):
    """
    Helper function to plot rose diagram on a given axis.
    """
    if len(results_df) == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                ha='center', va='center', fontsize=14)
        return
    
    fast_directions = []
    for phi in results_df['phi'].values:  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean
    original_directions = np.deg2rad(results_df['phi'].values)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)


def plot_fast_direction_rose_360(results_df, title="Fast Direction Distribution", 
                                  nbins=36, figsize=(8, 8), color='steelblue',
                                  edgecolor='black', linewidth=0.5):
    """
    Create a 360° polar rose plot (histogram) of fast directions with 180° symmetry.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' column
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for phi in results_df['phi'].values:  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit (counts are doubled due to symmetry)
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean for original ±90° range
    original_directions = np.deg2rad(results_df['phi'].values)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax

In [ ]:
# Load results
results_df = pd.read_csv('../results/splitting_results_axec2_apr_20_28.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df,
    title=f"Fast Direction Rose Plot - Station AXEC2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df,
    figsize=(12,8),
    station='AXEC2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=1,   # 2 samples
    sigma=1.5,
)
plt.show()

In [ ]:
# Load results
results_df_002_tmid_1_9_2_1 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_002_tmid_1_9_2_1,
    title=f"Fast Direction Rose Plot - Station AXEC2, SWSPy: 0.02s to S-pick Start, 1.9-2.1 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_002_tmid_1_9_2_1,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 0.02s to S-pick Start, 1.9-2.1 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_002_tmid_1_9_2_1,
    figsize=(12,8),
    station='AXEC2, SWSPy: 0.02s to S-pick Start, 1.9-2.1 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=1,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_002_tmid_1_5_2_5 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_5_2_5.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_002_tmid_1_5_2_5,
    title=f"Fast Direction Rose Plot - Station AXEC2, SWSPy: 0.02s to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_002_tmid_1_5_2_5,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 0.02s to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_002_tmid_1_5_2_5,
    figsize=(12,8),
    station='AXEC2, SWSPy: 0.02s to S-pick Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=1,   # 2 samples
    sigma=2,
)
plt.show()